## Cell 0 — Environment setup (run first, before anything else)



In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

print("Thread-limiting env vars set. Now install packages (skip any you already have).")


In [ ]:
!pip install --quiet biopython scikit-learn scipy pandas numpy xgboost shap lime


!pip install --quiet torch --index-url https://download.pytorch.org/whl/cpu
!pip install --quiet fair-esm

print("If any install failed or you skipped ESM/torch, that's fine -- set")
print("use_esm_embeddings=False in Cell 17 and everything else still runs.")


## Cell 1 — Imports & Environment Setup

In [ ]:
import argparse
import json
import math
import pickle
import warnings
import hashlib
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from Bio.Align import substitution_matrices
from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils.ProtParamData import kd
from scipy.stats import loguniform, randint, t
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    confusion_matrix, f1_score, matthews_corrcoef, precision_recall_curve, roc_auc_score,
)
from sklearn.model_selection import (
    GroupKFold, RandomizedSearchCV, RepeatedStratifiedKFold,
    StratifiedGroupKFold, StratifiedKFold, cross_val_predict,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, NuSVC

import sklearn
_SKLEARN_VERSION = tuple(map(int, sklearn.__version__.split('.')[:2]))
_HISTGB_HAS_CLASS_WEIGHT = _SKLEARN_VERSION >= (1, 2)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    XGBClassifier = None
    HAS_XGB = False

try:
    import torch
    import esm
    HAS_ESM = True
except Exception:
    HAS_ESM = False

try:
    import shap
    HAS_SHAP = True
except Exception:
    HAS_SHAP = False

try:
    import lime
    import lime.lime_tabular
    HAS_LIME = True
except Exception:
    HAS_LIME = False

STANDARD_AA = "ACDEFGHIKLMNPQRSTVWY"
AA_SET = set(STANDARD_AA)
ALL_DIPEPTIDES = [a + b for a in STANDARD_AA for b in STANDARD_AA]
BLOSUM62 = substitution_matrices.load("BLOSUM62")

ESM_MODEL_INFO = {
    "esm2_t6_8M_UR50D": 6,
    "esm2_t12_35M_UR50D": 12,
    "esm2_t30_150M_UR50D": 30,
    "esm2_t33_650M_UR50D": 33,
}

print(f"sklearn: {sklearn.__version__} | XGB: {HAS_XGB} | ESM: {HAS_ESM} | SHAP: {HAS_SHAP} | LIME: {HAS_LIME}")


## Cell 2 — Configuration & Lookup Tables

In [ ]:
@dataclass
class Config:
    input_csv: str = "main_dataset_cleaned_2.csv"
    outdir: str = "ml_report_v5"
    seq_col: str = "Peptide"
    label_col: str = "label"
    id_col: str = "dataset_id"

    # MHC restriction
    mhc_rank_col: str = "MHC_Percentile_Rank"
    mhc_ic50_col: str = "MHC_IC50_nM"
    mhc_class_col: str = "MHC_Class_Used"
    mhc_allele_col: str = "MHC_Allele_Best"
    require_mhc_assay: bool = True
    require_known_class: bool = True


    require_tcell_assay: bool = True
    assay_col: str = "Assay"
    tcell_assay_keyword: str = "T cell assay"

    # Similarity / conflict
    similarity_threshold: float = 0.80
    conflict_consensus_fraction: float = 0.75
    resolve_conflicts: bool = True

    # Features
    include_dpc: bool = True
    include_anchor_features: bool = True
    autoc_maxlag: int = 5
    use_mhc_features: bool = True
    mhc_allele_min_count: int = 5

    # ESM-2
    use_esm_embeddings: bool = True
    esm_model_name: str = "esm2_t12_35M_UR50D"

    esm_repr_layer: int = -1
    esm_batch_size: int = 16
    esm_device: str = ""

    # Selection
    var_threshold: float = 1e-5
    corr_threshold: float = 0.90
    mi_top_k: int = 300

    # CV / HPO
    outer_folds: int = 5
    outer_repeats: int = 3
    inner_folds: int = 3
    use_similarity_cv: str = "auto"
    random_search_iter: int = 35      # CHANGED: was 25.
    threshold_metric: str = "mcc"
    ensemble_min_inner_auc: float = 0.70

    # NEW: explainability / publication artifacts
    save_explainability_artifacts: bool = True
    shap_max_background: int = 250
    lime_n_explanations: int = 3

    random_seed: int = 42
    n_jobs: int = 4

    def __post_init__(self):
        if self.esm_repr_layer is None or self.esm_repr_layer < 0:
            self.esm_repr_layer = ESM_MODEL_INFO.get(self.esm_model_name, 12)


CTD_GROUPS = {
    "Hydrophobicity": {1: set("RKEDQN"), 2: set("GASTPHY"), 3: set("CLVIMFW")},
    "NormVanDerWaalsVolume": {1: set("GASTPDC"), 2: set("NVEQIL"), 3: set("MHKFRYW")},
    "Polarity": {1: set("LIFWCMVY"), 2: set("PATGS"), 3: set("HQRKNED")},
    "Polarizability": {1: set("GASDT"), 2: set("CPNVEQIL"), 3: set("KMHFRYW")},
    "Charge": {1: set("KR"), 2: set("ANCQGHILMFPSTWYV"), 3: set("DE")},
    "SecondaryStructure": {1: set("EALMQKRH"), 2: set("VIYCWFT"), 3: set("GNPSD")},
    "SolventAccessibility": {1: set("ALFCGIVW"), 2: set("RKQEND"), 3: set("MSPTHY")},
}

AA_POLARITY_GRANTHAM = {
    "A": 8.1, "R": 10.5, "N": 11.6, "D": 13.0, "C": 5.5, "Q": 10.5, "E": 12.3, "G": 9.0,
    "H": 10.4, "I": 5.2, "L": 4.9, "K": 11.3, "M": 5.7, "F": 5.2, "P": 8.0, "S": 9.2,
    "T": 8.6, "W": 5.4, "Y": 6.2, "V": 5.9,
}

AA_VOLUME_ZAMYATNIN = {
    "A": 88.6, "R": 173.4, "N": 114.1, "D": 111.1, "C": 108.5, "Q": 143.8, "E": 138.4,
    "G": 60.1, "H": 153.2, "I": 166.7, "L": 166.7, "K": 168.6, "M": 162.9, "F": 189.9,
    "P": 112.7, "S": 89.0, "T": 116.1, "W": 227.8, "Y": 193.6, "V": 140.0,
}

HYDROPHOBIC_AA = set("AILMFWVC")


## Cell 3 — Sequence Utilities & Similarity Clustering

In [ ]:
def clean_sequence(seq: str) -> str:
    return str(seq).strip().upper()


def is_valid_peptide(seq: str) -> bool:
    seq = clean_sequence(seq)
    return len(seq) > 0 and set(seq).issubset(AA_SET)


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x

    def union(self, a: int, b: int):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.rank[ra] < self.rank[rb]:
            ra, rb = rb, ra
        self.parent[rb] = ra
        if self.rank[ra] == self.rank[rb]:
            self.rank[ra] += 1


def normalized_edit_identity(a: str, b: str, threshold_hint: float = 0.0) -> float:
    a, b = clean_sequence(a), clean_sequence(b)
    if a == b:
        return 1.0
    la, lb = len(a), len(b)
    max_len = max(la, lb)
    if max_len == 0:
        return 0.0
    if min(la, lb) / max_len < threshold_hint:
        return min(la, lb) / max_len
    prev = list(range(lb + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            curr.append(min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = curr
    return 1.0 - prev[-1] / max_len


def compute_similarity_groups(seqs: list, threshold: float) -> np.ndarray:
    """Union-find clusters of near-duplicate sequences (identity >= threshold)."""
    n = len(seqs)
    if n > 3000:
        print(f"[WARNING] O(n^2) similarity clustering on {n} sequences will be very slow.")
        print("          Consider pre-clustering with cd-hit or mmseqs for larger datasets.")

    uf = UnionFind(n)
    for i in range(n):
        if i % 250 == 0 and i > 0:
            print(f"[similarity] compared {i}/{n} sequences")
        for j in range(i + 1, n):
            if min(len(seqs[i]), len(seqs[j])) / max(len(seqs[i]), len(seqs[j])) < threshold:
                continue
            if normalized_edit_identity(seqs[i], seqs[j], threshold_hint=threshold) >= threshold:
                uf.union(i, j)
    roots = [uf.find(i) for i in range(n)]
    root_to_id = {root: idx for idx, root in enumerate(sorted(set(roots)))}
    return np.array([root_to_id[root] for root in roots])


## Cell 4a — Feature Extractors: AAC / DPC / CTD / Physicochemical

In [ ]:
def aac_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    counts = Counter(seq)
    length = len(seq)
    return {f"AAC_{aa}": counts.get(aa, 0) / length * 100 for aa in STANDARD_AA}


def dpc_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    denom = max(len(seq) - 1, 1)
    counts = Counter(seq[i:i + 2] for i in range(len(seq) - 1))
    return {f"DPC_{dp}": counts.get(dp, 0) / denom * 100 for dp in ALL_DIPEPTIDES}


def _ctd_group_of(aa: str, groups: dict):
    for group_id, members in groups.items():
        if aa in members:
            return group_id
    return None


def ctd_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    feats = {}
    for attr, groups in CTD_GROUPS.items():
        group_seq = [_ctd_group_of(aa, groups) for aa in seq]
        counts = Counter(group_seq)
        for gid in (1, 2, 3):
            feats[f"CTD_{attr}_C{gid}"] = counts.get(gid, 0) / length * 100
        denom = max(length - 1, 1)
        transitions = Counter()
        for i in range(length - 1):
            left, right = group_seq[i], group_seq[i + 1]
            if left is None or right is None or left == right:
                continue
            transitions[tuple(sorted((left, right)))] += 1
        for pair in [(1, 2), (1, 3), (2, 3)]:
            feats[f"CTD_{attr}_T{pair[0]}{pair[1]}"] = transitions.get(pair, 0) / denom * 100
        for gid in (1, 2, 3):
            positions = [i + 1 for i, g in enumerate(group_seq) if g == gid]
            if not positions:
                distribution = [0.0] * 5
            else:
                n = len(positions)
                idxs = [1, max(1, math.ceil(0.25 * n)), max(1, math.ceil(0.50 * n)),
                        max(1, math.ceil(0.75 * n)), n]
                distribution = [positions[idx - 1] / length * 100 for idx in idxs]
            for pct, val in zip([0, 25, 50, 75, 100], distribution):
                feats[f"CTD_{attr}_D{gid}_{pct}"] = val
    return feats


def physicochemical_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        values = np.array([table[aa] for aa in seq], dtype=float)
        feats[f"PHYS_{name}_mean"] = float(values.mean())
        feats[f"PHYS_{name}_std"] = float(values.std())
    analysis = ProteinAnalysis(seq)
    feats["PHYS_molecular_weight"] = analysis.molecular_weight()
    feats["PHYS_aromaticity"] = analysis.aromaticity()
    feats["PHYS_instability_index"] = analysis.instability_index()
    try:
        feats["PHYS_isoelectric_point"] = analysis.isoelectric_point()
    except Exception:
        feats["PHYS_isoelectric_point"] = np.nan
    feats["PHYS_gravy"] = analysis.gravy()
    feats["PHYS_charge_at_pH7"] = analysis.charge_at_pH(7.0)
    helix, turn, sheet = analysis.secondary_structure_fraction()
    feats["PHYS_ss_fraction_helix"] = helix
    feats["PHYS_ss_fraction_turn"] = turn
    feats["PHYS_ss_fraction_sheet"] = sheet
    feats["PHYS_length"] = len(seq)
    return feats


## Cell 4b — Feature Extractors: DHKR / Autocorrelation / BLOSUM / Terminal / Anchor

In [ ]:
def dhkr_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    length = len(seq)
    counts = Counter(seq)
    feats = {f"DHKR_frac_{aa}": counts.get(aa, 0) / length * 100 for aa in "DHKRE"}
    feats["DHKR_charged_fraction"] = sum(counts.get(aa, 0) for aa in "DHKRE") / length * 100
    feats["DHKR_net_charge_proxy"] = (
        counts.get("K", 0) + counts.get("R", 0) + 0.1 * counts.get("H", 0)
        - counts.get("D", 0) - counts.get("E", 0)
    ) / length
    return feats


def _moran_autocorrelation(seq: str, table: dict, maxlag: int) -> list:
    values = np.array([table[aa] for aa in clean_sequence(seq)], dtype=float)
    length = len(values)
    mean_value = values.mean()
    denom = np.sum((values - mean_value) ** 2) / length
    out = []
    for lag in range(1, maxlag + 1):
        if lag >= length or denom == 0:
            out.append(0.0)
            continue
        numerator = np.sum((values[:-lag] - mean_value) * (values[lag:] - mean_value)) / (length - lag)
        out.append(float(numerator / denom))
    return out


def autocorrelation_features(seq: str, maxlag: int = 5) -> dict:
    seq = clean_sequence(seq)
    maxlag = max(0, min(maxlag, len(seq) - 1))
    feats = {}
    for name, table in [("Hydrophobicity_KD", kd), ("Polarity_Grantham", AA_POLARITY_GRANTHAM),
                         ("Volume_Zamyatnin", AA_VOLUME_ZAMYATNIN)]:
        for lag, value in enumerate(_moran_autocorrelation(seq, table, maxlag), start=1):
            feats[f"AUTOC_Moran_{name}_lag{lag}"] = value
    return feats


def blosum62_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    acc = {aa: 0.0 for aa in STANDARD_AA}
    for residue in seq:
        for aa in STANDARD_AA:
            acc[aa] += BLOSUM62[residue][aa]
    return {f"BLOSUM_{aa}": acc[aa] / len(seq) for aa in STANDARD_AA}


def terminal_composition_features(seq: str) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    for side, window in [("N", seq[:2]), ("C", seq[-2:]), ("N3", seq[:3]), ("C3", seq[-3:])]:
        counts = Counter(window)
        denom = max(len(window), 1)
        for aa in STANDARD_AA:
            feats[f"TERM_{side}_AAC_{aa}"] = counts.get(aa, 0) / denom * 100
    return feats


def anchor_position_features(seq: str, mhc_class: str) -> dict:

    seq = clean_sequence(seq)
    L = len(seq)
    mhc_class = str(mhc_class).strip().upper() if mhc_class is not None else ""
    feats = {
        "ANCHOR_P2_hydro": 0.0, "ANCHOR_Pomega_hydro": 0.0,
        "ANCHOR_P2_is_hydrophobic": 0.0, "ANCHOR_Pomega_is_hydrophobic": 0.0,
        "ANCHOR_classII_a1_hydro": 0.0, "ANCHOR_classII_a4_hydro": 0.0,
        "ANCHOR_classII_a6_hydro": 0.0, "ANCHOR_classII_a9_hydro": 0.0,
    }
    if L == 0:
        return feats
    if mhc_class == "I":
        p2 = seq[1] if L > 1 else seq[0]
        pomega = seq[-1]
        feats["ANCHOR_P2_hydro"] = float(kd.get(p2, 0.0))
        feats["ANCHOR_Pomega_hydro"] = float(kd.get(pomega, 0.0))
        feats["ANCHOR_P2_is_hydrophobic"] = float(p2 in HYDROPHOBIC_AA)
        feats["ANCHOR_Pomega_is_hydrophobic"] = float(pomega in HYDROPHOBIC_AA)
    elif mhc_class == "II":
        idxs = [max(0, min(L - 1, int(round(f * (L - 1))))) for f in (0.0, 0.33, 0.55, 0.9)]
        keys = ["ANCHOR_classII_a1_hydro", "ANCHOR_classII_a4_hydro",
                "ANCHOR_classII_a6_hydro", "ANCHOR_classII_a9_hydro"]
        for key, idx in zip(keys, idxs):
            feats[key] = float(kd.get(seq[idx], 0.0))
    return feats


def extract_all_features(seq: str, cfg: Config, mhc_class: str = None) -> dict:
    seq = clean_sequence(seq)
    feats = {}
    feats.update(aac_features(seq))
    if cfg.include_dpc:
        feats.update(dpc_features(seq))
    feats.update(ctd_features(seq))
    feats.update(physicochemical_features(seq))
    feats.update(dhkr_features(seq))
    feats.update(autocorrelation_features(seq, maxlag=cfg.autoc_maxlag))
    feats.update(blosum62_features(seq))
    feats.update(terminal_composition_features(seq))
    if cfg.include_anchor_features:
        feats.update(anchor_position_features(seq, mhc_class))
    return feats


def feature_family(feature_name: str) -> str:
    for prefix, fam in [("AAC_", "AAC"), ("DPC_", "DPC"), ("CTD_", "CTD"), ("PHYS_", "PHYS"),
                         ("DHKR_", "DHKR"), ("AUTOC_", "AUTOC"), ("BLOSUM_", "BLOSUM"),
                         ("TERM_", "TERMINAL"), ("ANCHOR_", "ANCHOR"),
                         ("ESM2_", "ESM2"), ("MHC_", "MHC")]:
        if feature_name.startswith(prefix):
            return fam
    return "OTHER"


## Cell 5 — MHC Features & ESM-2

In [ ]:
def extract_mhc_features(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    mhc = pd.DataFrame(index=df.index)
    rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
    ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")

    mhc["MHC_rank_missing"] = rank.isna().astype(float)
    mhc["MHC_ic50_missing"] = ic50.isna().astype(float)

    approx_rank_from_ic50 = np.clip(np.log10(ic50 + 1.0) * 2.5, 0, 100)
    approx_ic50_from_rank = np.clip(10 ** (rank / 2.5), 1, 100000)
    mhc["MHC_rank"] = rank.fillna(approx_rank_from_ic50).fillna(100.0)
    mhc["MHC_ic50"] = ic50.fillna(approx_ic50_from_rank).fillna(50000.0)
    mhc["MHC_log10_ic50"] = np.log10(mhc["MHC_ic50"] + 1.0)
    mhc["MHC_sqrt_rank"] = np.sqrt(mhc["MHC_rank"])

    mhc["MHC_strong_binder"] = ((mhc["MHC_rank"] < 0.5) | (mhc["MHC_ic50"] < 50.0)).astype(float)
    mhc["MHC_good_binder"] = (((mhc["MHC_rank"] >= 0.5) & (mhc["MHC_rank"] < 2.0)) |
                               ((mhc["MHC_ic50"] >= 50.0) & (mhc["MHC_ic50"] < 500.0))).astype(float)
    mhc["MHC_weak_binder"] = (((mhc["MHC_rank"] >= 2.0) & (mhc["MHC_rank"] < 10.0)) |
                               ((mhc["MHC_ic50"] >= 500.0) & (mhc["MHC_ic50"] < 5000.0))).astype(float)
    mhc["MHC_non_binder"] = ((mhc["MHC_rank"] >= 10.0) & (mhc["MHC_ic50"] >= 5000.0)).astype(float)
    mhc["MHC_binding_score"] = np.clip(1.0 / (1.0 + mhc["MHC_rank"]), 0, 1)
    mhc["MHC_log_ic50_score"] = np.clip(1.0 - (np.log10(mhc["MHC_ic50"] + 1) / 5.0), 0, 1)

    class_used = df.get(cfg.mhc_class_col, pd.Series("unknown", index=df.index)).astype(str).str.strip().str.upper()
    mhc["MHC_class_I"] = (class_used == "I").astype(float)
    mhc["MHC_class_II"] = (class_used == "II").astype(float)

    length = df[cfg.seq_col].astype(str).str.len()
    mhc["MHC_classI_x_short"] = mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
    mhc["MHC_classII_x_long"] = mhc["MHC_class_II"] * ((length >= 13) & (length <= 25)).astype(float)
    mhc["MHC_compatible_length"] = (
        mhc["MHC_class_I"] * ((length >= 8) & (length <= 11)).astype(float)
        + mhc["MHC_class_II"] * ((length >= 12) & (length <= 25)).astype(float)
    )

    if cfg.mhc_allele_col in df.columns:
        allele = df[cfg.mhc_allele_col].fillna("unknown").astype(str).str.strip()

        def allele_supertype(a):
            a = str(a).upper()
            for prefix, fam in [("HLA-A", "HLA_A"), ("HLA-B", "HLA_B"), ("HLA-C", "HLA_C"),
                                 ("HLA-DR", "HLA_DR"), ("HLA-DQ", "HLA_DQ"), ("HLA-DP", "HLA_DP")]:
                if a.startswith(prefix):
                    return fam
            return "Other"

        fam = allele.apply(allele_supertype)
        fam_counts = fam.value_counts()
        valid_fams = fam_counts[fam_counts >= cfg.mhc_allele_min_count].index.tolist()
        for f in valid_fams:
            mhc[f"MHC_allele_fam_{f}"] = (fam == f).astype(float)

    return mhc.fillna(0.0)


def _esm_cache_path(seqs: list, cfg: Config) -> Path:
    sig = f"{cfg.esm_model_name}|{cfg.esm_repr_layer}|{len(seqs)}|{seqs[0]}|{seqs[-1]}"
    h = hashlib.md5(sig.encode()).hexdigest()[:16]
    return Path(f".esm_cache_{h}.pkl")


def esm2_embedding_features(seqs: list, cfg: Config) -> pd.DataFrame:
    if not cfg.use_esm_embeddings:
        return pd.DataFrame(index=pd.RangeIndex(len(seqs)))
    if not HAS_ESM:
        raise RuntimeError(
            "ESM-2 requires 'torch' and 'fair-esm'. Install with:\n"
            "  pip install torch --index-url https://download.pytorch.org/whl/cpu\n"
            "  pip install fair-esm"
        )
    if not seqs:
        return pd.DataFrame(index=pd.RangeIndex(0))

    cache_path = _esm_cache_path(seqs, cfg)
    if cache_path.exists():
        print(f"[ESM-2] Loading cached embeddings from {cache_path}")
        return pd.read_pickle(cache_path)

    device = cfg.esm_device.strip() if cfg.esm_device else ("cuda" if torch.cuda.is_available() else "cpu")
    if not hasattr(esm.pretrained, cfg.esm_model_name):
        raise ValueError(f"Unknown ESM model '{cfg.esm_model_name}'.")

    print(f"[ESM-2] Loading {cfg.esm_model_name} (layer {cfg.esm_repr_layer}) on {device}.")
    model, alphabet = getattr(esm.pretrained, cfg.esm_model_name)()
    model = model.eval().to(device)
    batch_converter = alphabet.get_batch_converter()

    records, feature_names, total = [], None, len(seqs)
    with torch.no_grad():
        for start in range(0, total, cfg.esm_batch_size):
            batch_seqs = seqs[start:start + cfg.esm_batch_size]
            batch = [(str(start + i), seq) for i, seq in enumerate(batch_seqs)]
            _, strings, tokens = batch_converter(batch)
            tokens = tokens.to(device)
            output = model(tokens, repr_layers=[cfg.esm_repr_layer], return_contacts=False)
            reps = output["representations"][cfg.esm_repr_layer].detach().cpu().numpy()
            for row_idx, seq in enumerate(strings):
                vector = reps[row_idx, 1:len(seq) + 1].mean(axis=0)
                if feature_names is None:
                    feature_names = [f"ESM2_{i:03d}" for i in range(vector.shape[0])]
                records.append(vector.astype(float))
            print(f"[ESM-2] embedded {min(start + len(batch_seqs), total)}/{total} peptides")

    out_df = pd.DataFrame(records, columns=feature_names)
    out_df.to_pickle(cache_path)
    print(f"[ESM-2] Saved embeddings to {cache_path}")
    return out_df


## Cell 6 — Build Feature Matrix

In [ ]:
def build_feature_matrix(df: pd.DataFrame, cfg: Config) -> pd.DataFrame:
    """Assumes df has already been filtered to valid sequences upstream."""
    clean_list = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    if len(clean_list) == 0:
        raise ValueError("No peptides remain after upstream filtering.")

    if cfg.include_anchor_features and cfg.mhc_class_col in df.columns:
        mhc_class_list = df[cfg.mhc_class_col].astype(str).tolist()
    else:
        mhc_class_list = [None] * len(clean_list)

    records = [extract_all_features(seq, cfg, mhc_class=cls)
               for seq, cls in zip(clean_list, mhc_class_list)]
    feat_df = pd.DataFrame.from_records(records).fillna(0.0)

    esm_df = esm2_embedding_features(clean_list, cfg).fillna(0.0)
    feat_df = pd.concat([feat_df.reset_index(drop=True), esm_df.reset_index(drop=True)], axis=1)

    if cfg.use_mhc_features:
        mhc_df = extract_mhc_features(df, cfg).reset_index(drop=True)
        feat_df = pd.concat([feat_df, mhc_df], axis=1)
        print(f"[feature extraction] Added {mhc_df.shape[1]} MHC features.")

    lead_cols = {}
    if cfg.id_col and cfg.id_col in df.columns:
        lead_cols[cfg.id_col] = df[cfg.id_col].values
    lead_cols[cfg.label_col] = df[cfg.label_col].values
    lead_cols[cfg.seq_col] = df[cfg.seq_col].values

    print(f"[feature extraction] Built {feat_df.shape[1]} features "
          f"(DPC {'ON' if cfg.include_dpc else 'OFF'}, "
          f"anchor {'ON' if cfg.include_anchor_features else 'OFF'}) "
          f"for {feat_df.shape[0]} peptides.")
    return pd.concat([pd.DataFrame(lead_cols).reset_index(drop=True), feat_df.reset_index(drop=True)], axis=1)


## Cell 7 — Assay Restriction, Conflict Resolution

In [ ]:
def restrict_to_assay_supported(df: pd.DataFrame, cfg: Config) -> tuple:
    n_before = len(df)
    keep = pd.Series(True, index=df.index)
    reasons = {}

    if cfg.require_mhc_assay:
        rank = pd.to_numeric(df.get(cfg.mhc_rank_col), errors="coerce")
        ic50 = pd.to_numeric(df.get(cfg.mhc_ic50_col), errors="coerce")
        has_assay = rank.notna() | ic50.notna()
        reasons["dropped_no_assay_value"] = int((~has_assay).sum())
        keep &= has_assay

    if cfg.require_known_class:
        class_used = df.get(cfg.mhc_class_col, pd.Series(index=df.index, dtype=object))
        class_used = class_used.astype(str).str.strip().str.upper()
        has_class = class_used.isin(["I", "II"])
        reasons["dropped_unknown_class"] = int((~has_class & keep).sum())
        keep &= has_class

    out = df.loc[keep].reset_index(drop=True)
    report = {"n_before": n_before, "n_after": int(len(out)), **reasons}
    print(f"[restrict] {n_before} -> {len(out)} peptides after MHC assay-support restriction: {reasons}")
    return out, report


def restrict_to_tcell_assay(df: pd.DataFrame, cfg: Config) -> tuple:

    if not cfg.require_tcell_assay or cfg.assay_col not in df.columns:
        return df, {"require_tcell_assay": False}
    n_before = len(df)
    has_tcell = df[cfg.assay_col].astype(str).str.contains(cfg.tcell_assay_keyword, na=False, case=False)
    out = df.loc[has_tcell].reset_index(drop=True)
    report = {
        "n_before_tcell_filter": n_before,
        "n_after_tcell_filter": int(len(out)),
        "n_dropped_no_tcell_evidence": int((~has_tcell).sum()),
    }
    print(f"[restrict] T-cell-assay filter: {n_before} -> {len(out)} peptides "
          f"({report['n_dropped_no_tcell_evidence']} dropped -- no direct T-cell assay evidence).")
    return out, report


def resolve_label_conflicts(df: pd.DataFrame, groups: np.ndarray, cfg: Config) -> tuple:
    y = df[cfg.label_col].to_numpy().astype(int)
    resolved = y.copy()
    drop_mask = np.zeros(len(df), dtype=bool)
    n_relabeled_groups, n_relabeled_peptides = 0, 0
    n_dropped_groups, n_dropped_peptides = 0, 0

    for g in np.unique(groups):
        idx = np.where(groups == g)[0]
        if len(idx) <= 1:
            continue
        labels = y[idx]
        if len(set(labels.tolist())) == 1:
            continue
        pos_frac = float((labels == 1).mean())
        if pos_frac >= cfg.conflict_consensus_fraction:
            changed = idx[labels == 0]
            resolved[idx] = 1
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        elif pos_frac <= (1.0 - cfg.conflict_consensus_fraction):
            changed = idx[labels == 1]
            resolved[idx] = 0
            n_relabeled_groups += 1
            n_relabeled_peptides += len(changed)
        else:
            drop_mask[idx] = True
            n_dropped_groups += 1
            n_dropped_peptides += len(idx)

    out = df.copy()
    out[cfg.label_col] = resolved
    out = out.loc[~drop_mask].reset_index(drop=True)
    kept_groups = groups[~drop_mask]
    report = {
        "n_conflicting_groups_relabeled": n_relabeled_groups,
        "n_peptides_relabeled_to_consensus": n_relabeled_peptides,
        "n_conflicting_groups_dropped": n_dropped_groups,
        "n_peptides_dropped_as_contradictory": n_dropped_peptides,
        "n_peptides_after_resolution": int(len(out)),
    }
    print(f"[conflict resolution] {report}")
    return out, kept_groups, report


## Cell 8 — Feature Selection Transformers

In [ ]:
def remove_low_variance(X_train: pd.DataFrame, threshold: float) -> list:
    variances = X_train.var(axis=0)
    return variances[variances > threshold].index.tolist()


def remove_highly_correlated(X_train: pd.DataFrame, threshold: float) -> list:
    corr = X_train.corr().abs()
    upper = corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
    to_drop = {col for col in upper.columns if any(upper[col] > threshold)}
    return [c for c in X_train.columns if c not in to_drop]


class VarianceCorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, var_threshold: float = 1e-5, corr_threshold: float = 0.90):
        self.var_threshold = var_threshold
        self.corr_threshold = corr_threshold

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        cols_var = remove_low_variance(X_df, self.var_threshold)
        self.selected_columns_ = remove_highly_correlated(X_df[cols_var], self.corr_threshold)
        if not self.selected_columns_:
            self.selected_columns_ = cols_var[:1] if cols_var else self.feature_names_in_[:1]
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class MutualInfoTopKSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k: int = 250, random_state: int = 42, force_include_prefix: str = "MHC_"):
        self.k = k
        self.random_state = random_state
        self.force_include_prefix = force_include_prefix

    def fit(self, X, y):
        X_df = pd.DataFrame(X).copy()
        self.feature_names_in_ = X_df.columns.astype(str).tolist()
        X_df.columns = self.feature_names_in_
        forced = [c for c in self.feature_names_in_ if c.startswith(self.force_include_prefix)]
        remaining = [c for c in self.feature_names_in_ if c not in forced]
        if remaining and self.k and self.k > 0:
            n_from_mi = max(0, self.k - len(forced))
            if 0 < n_from_mi < len(remaining):
                mi = mutual_info_classif(X_df[remaining], y, random_state=self.random_state)
                order = np.argsort(mi)[::-1][:n_from_mi]
                selected_from_mi = [remaining[i] for i in order]
            else:
                selected_from_mi = remaining
        else:
            selected_from_mi = remaining
        self.selected_columns_ = forced + selected_from_mi
        if not self.selected_columns_:
            self.selected_columns_ = self.feature_names_in_
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X).copy()
        X_df.columns = self.feature_names_in_
        return X_df[self.selected_columns_]


class WeightedSoftVotingEnsemble(BaseEstimator, ClassifierMixin):
    def __init__(self, estimators: dict, weights: np.ndarray, threshold: float = 0.5):
        self.estimators = estimators
        self.weights = weights
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def fit(self, X, y=None):
        return self

    def predict_proba(self, X):
        probs = np.array([est.predict_proba(X)[:, 1] for est in self.estimators.values()])
        pos = np.average(probs, axis=0, weights=self.weights)
        return np.column_stack([1.0 - pos, pos])

    def predict(self, X):
        return (self.predict_proba(X)[:, 1] >= self.threshold).astype(int)


def make_model_pipeline(estimator, cfg: Config) -> Pipeline:
    return Pipeline(steps=[
        ("var_corr", VarianceCorrelationFilter(cfg.var_threshold, cfg.corr_threshold)),
        ("mi", MutualInfoTopKSelector(k=cfg.mi_top_k, random_state=cfg.random_seed)),
        ("scaler", StandardScaler()),
        ("classifier", estimator),
    ])


def pipeline_selected_features(estimator) -> list:
    if not isinstance(estimator, Pipeline):
        return []
    try:
        return estimator.named_steps["mi"].selected_columns_
    except Exception:
        return []


def optimize_threshold(y_true: np.ndarray, proba: np.ndarray, metric: str) -> float:
    precision, recall, thresholds = precision_recall_curve(y_true, proba)
    if len(thresholds) == 0:
        return 0.5
    candidates = np.unique(np.concatenate([thresholds, np.linspace(0.10, 0.90, 81)]))
    best_threshold, best_score = 0.5, -np.inf
    for threshold in candidates:
        pred = (proba >= threshold).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        elif metric == "balanced_accuracy":
            score = balanced_accuracy_score(y_true, pred)
        else:
            score = matthews_corrcoef(y_true, pred)
        if score > best_score:
            best_score, best_threshold = score, float(threshold)
    return best_threshold


## Cell 9 — Model Search Space, Tuning & Ensemble

In [ ]:
def model_search_space(y_train: np.ndarray, cfg: Config) -> dict:
    n_pos, n_neg = int((y_train == 1).sum()), int((y_train == 0).sum())
    scale_pos_weight = n_neg / max(n_pos, 1)
    space = {
        "logreg_l2": (make_model_pipeline(
            LogisticRegression(penalty="l2", solver="lbfgs", max_iter=6000,
                                class_weight="balanced", random_state=cfg.random_seed), cfg),
            {"classifier__C": loguniform(1e-3, 1e2)}),
        "svm_rbf": (make_model_pipeline(
            SVC(kernel="rbf", probability=True, class_weight="balanced",
                random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__C": loguniform(1e-2, 1e2), "classifier__gamma": loguniform(1e-4, 1e0)}),
        "nu_svm": (make_model_pipeline(
            NuSVC(kernel="rbf", gamma="scale", probability=True,
                  random_state=cfg.random_seed, cache_size=1000), cfg),
            {"classifier__nu": [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8],
             "classifier__gamma": loguniform(1e-4, 1e0)}),
        "rf": (make_model_pipeline(
            RandomForestClassifier(random_state=cfg.random_seed, n_jobs=1,
                                    class_weight="balanced_subsample"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 10), "classifier__min_samples_split": randint(2, 16),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
        "extra_trees": (make_model_pipeline(
            ExtraTreesClassifier(random_state=cfg.random_seed, n_jobs=1, class_weight="balanced"), cfg),
            {"classifier__n_estimators": randint(300, 900), "classifier__max_depth": [3, 5, 8, 12, 16, None],
             "classifier__min_samples_leaf": randint(1, 8), "classifier__min_samples_split": randint(2, 14),
             "classifier__max_features": ["sqrt", "log2", 0.25, 0.4, 0.6]}),
    }

    hist_kwargs = {"random_state": cfg.random_seed}
    if _HISTGB_HAS_CLASS_WEIGHT:
        hist_kwargs["class_weight"] = "balanced"

    space["hist_gb"] = (make_model_pipeline(
        HistGradientBoostingClassifier(**hist_kwargs), cfg),
        {"classifier__max_iter": randint(100, 500), "classifier__max_leaf_nodes": randint(7, 63),
         "classifier__learning_rate": loguniform(0.01, 0.25),
         "classifier__l2_regularization": loguniform(1e-4, 20),
         "classifier__min_samples_leaf": randint(10, 60)})

    if HAS_XGB:
        space["xgb"] = (make_model_pipeline(
            XGBClassifier(random_state=cfg.random_seed, n_jobs=1, eval_metric="auc",
                        tree_method="hist", scale_pos_weight=scale_pos_weight), cfg),
            {"classifier__n_estimators": randint(150, 700), "classifier__max_depth": randint(2, 7),
             "classifier__learning_rate": loguniform(0.01, 0.25), "classifier__subsample": [0.60, 0.75, 0.90, 1.0],
             "classifier__colsample_bytree": [0.50, 0.65, 0.80, 1.0], "classifier__min_child_weight": randint(1, 12),
             "classifier__reg_alpha": loguniform(1e-4, 10), "classifier__reg_lambda": loguniform(1e-3, 30)})
    return space


def tune_models(X_train, y_train, inner_cv, cfg: Config, groups_train=None) -> dict:
    tuned = {}
    for name, (estimator, params) in model_search_space(y_train, cfg).items():
        search = RandomizedSearchCV(estimator=estimator, param_distributions=params, n_iter=cfg.random_search_iter,
                                     scoring="roc_auc", cv=inner_cv, random_state=cfg.random_seed,
                                     n_jobs=cfg.n_jobs, refit=True, error_score=np.nan)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if groups_train is None:
                search.fit(X_train, y_train)
            else:
                search.fit(X_train, y_train, groups=groups_train)
        best_estimator = search.best_estimator_
        try:
            cv_kwargs = {"groups": groups_train} if groups_train is not None else {}
            oof_proba = cross_val_predict(clone(best_estimator), X_train, y_train, cv=inner_cv,
                                           method="predict_proba", n_jobs=cfg.n_jobs, **cv_kwargs)[:, 1]
            threshold = optimize_threshold(y_train, oof_proba, cfg.threshold_metric)
        except Exception:
            threshold = 0.5
        tuned[name] = {
            "estimator": best_estimator, "inner_cv_auc": float(search.best_score_),
            "best_params": search.best_params_, "threshold": threshold,
            "selected_features": pipeline_selected_features(best_estimator),
        }
    return tuned


def ensemble_member_items(tuned: dict, min_inner_auc: float) -> list:
    eligible = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"]) and i["inner_cv_auc"] >= min_inner_auc]
    if eligible:
        return eligible
    finite = [(n, i) for n, i in tuned.items() if np.isfinite(i["inner_cv_auc"])]
    return [max(finite, key=lambda item: item[1]["inner_cv_auc"])] if finite else list(tuned.items())


def ensemble_weights(member_items: list) -> np.ndarray:
    weights = np.array([max(info["inner_cv_auc"], 1e-6) for _, info in member_items], dtype=float)
    return weights / weights.sum()


def soft_vote_proba(tuned: dict, X: pd.DataFrame, min_inner_auc: float) -> np.ndarray:
    members = ensemble_member_items(tuned, min_inner_auc)
    weights = ensemble_weights(members)
    probs = np.array([info["estimator"].predict_proba(X)[:, 1] for _, info in members])
    return np.average(probs, axis=0, weights=weights)


def metric_row(y_true: np.ndarray, proba: np.ndarray, threshold: float) -> dict:
    # (patch v2) suppress sklearn's single-class UserWarning noise; behavior

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=UserWarning)
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
        specificity = tn / (tn + fp) if (tn + fp) else 0.0
        roc_auc = roc_auc_score(y_true, proba) if len(np.unique(y_true)) == 2 else np.nan
        pr_auc = average_precision_score(y_true, proba) if (y_true == 1).any() else np.nan
        result = {"roc_auc": roc_auc, "pr_auc": pr_auc, "accuracy": accuracy_score(y_true, pred),
                  "f1": f1_score(y_true, pred, zero_division=0), "mcc": matthews_corrcoef(y_true, pred),
                  "sensitivity": sensitivity, "specificity": specificity,
                  "balanced_accuracy": balanced_accuracy_score(y_true, pred), "threshold": threshold,
                  "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn)}
    return result


## Cell 10a — Cross-Validation Splitting & Nested CV Loop

In [ ]:
def make_outer_splits(y: np.ndarray, groups: np.ndarray, cfg: Config):

    use_groups = cfg.use_similarity_cv in ("always", "auto")
    if not use_groups:
        splitter = RepeatedStratifiedKFold(n_splits=cfg.outer_folds, n_repeats=cfg.outer_repeats,
                                            random_state=cfg.random_seed)
        for split_id, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y), start=1):
            repeat = (split_id - 1) // cfg.outer_folds + 1
            fold = (split_id - 1) % cfg.outer_folds + 1
            yield split_id, repeat, fold, train_idx, val_idx, "repeated_stratified_kfold"
        return

    n_unique_groups = len(np.unique(groups)) if groups is not None else 0
    effective_splits = min(cfg.outer_folds, n_unique_groups)
    if effective_splits < 2:
        raise ValueError(
            f"Cannot run group-aware CV: only {n_unique_groups} unique similarity "
            f"group(s) in this data but need >=2. Either this subset is too small/"
            f"too similarity-collapsed for group-aware CV, or set "
            f"use_similarity_cv='never' deliberately for this specific probe."
        )
    if effective_splits < cfg.outer_folds:
        print(f"[cv] WARNING: only {n_unique_groups} unique similarity groups available; "
              f"reducing outer_folds from {cfg.outer_folds} to {effective_splits} for this "
              f"run to avoid empty validation folds.")

    try:
        for repeat in range(1, cfg.outer_repeats + 1):
            splitter = StratifiedGroupKFold(n_splits=effective_splits, shuffle=True,
                                             random_state=cfg.random_seed + repeat - 1)
            for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
                if len(val_idx) == 0 or len(train_idx) == 0:
                    print(f"[cv] Skipping degenerate fold (repeat {repeat}, fold {fold}) "
                          f"with an empty train/val partition.")
                    continue
                split_id = (repeat - 1) * effective_splits + fold
                yield split_id, repeat, fold, train_idx, val_idx, "stratified_group_kfold_similarity_aware"
    except ValueError:
        raise
    except Exception as exc:
        print(f"[cv] StratifiedGroupKFold unavailable/failed ({exc}); falling back to GroupKFold.")
        splitter = GroupKFold(n_splits=effective_splits)
        for fold, (train_idx, val_idx) in enumerate(splitter.split(np.zeros(len(y)), y, groups=groups), start=1):
            if len(val_idx) == 0 or len(train_idx) == 0:
                continue
            yield fold, 1, fold, train_idx, val_idx, "group_kfold_similarity_aware"


def make_inner_cv(y_train: np.ndarray, groups_train: np.ndarray, cfg: Config, cv_mode: str):
    class_counts = np.bincount(y_train.astype(int), minlength=2)
    n_splits = min(cfg.inner_folds, int(class_counts.min()))
    if n_splits < 2:
        raise ValueError("Inner CV needs >=2 samples per class in the outer-training fold.")
    if "group" in cv_mode and groups_train is not None and len(np.unique(groups_train)) >= n_splits:
        sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed)
        try:
            next(sgkf.split(np.zeros(len(y_train)), y_train, groups=groups_train))
            return sgkf, groups_train
        except Exception as exc:
            print(f"[cv] StratifiedGroupKFold failed for inner CV ({exc}); falling back to StratifiedKFold.")
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), None


def run_nested_cv(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config,
                   row_ids: np.ndarray = None) -> tuple:

    if row_ids is None:
        row_ids = np.arange(len(y))

    rows, full_records, oof_records = [], [], []
    for split_id, repeat, fold, train_idx, val_idx, cv_mode in make_outer_splits(y, groups, cfg):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
        groups_train = groups[train_idx] if groups is not None else None
        inner_cv, inner_groups = make_inner_cv(y_train, groups_train, cfg, cv_mode)
        tuned = tune_models(X_train, y_train, inner_cv, cfg, groups_train=inner_groups)

        split_rows = []
        for model_name, info in tuned.items():
            proba = info["estimator"].predict_proba(X_val)[:, 1]
            metrics = metric_row(y_val, proba, info["threshold"])
            row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": model_name,
                   "n_features_final": len(info["selected_features"]), "inner_cv_auc": info["inner_cv_auc"], **metrics}
            split_rows.append(row)
            full_records.append({"row": row, "selected_features": info["selected_features"],
                                  "best_params": info["best_params"]})
            for i, ridx in enumerate(val_idx):
                oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": model_name,
                                     "proba": float(proba[i]), "y_true": int(y_val[i])})

        members = ensemble_member_items(tuned, cfg.ensemble_min_inner_auc)
        ens_proba = soft_vote_proba(tuned, X_val, cfg.ensemble_min_inner_auc)
        ens_threshold = optimize_threshold(y_val, ens_proba, cfg.threshold_metric)
        ens_metrics = metric_row(y_val, ens_proba, ens_threshold)
        ens_selected = sorted({f for _, info in members for f in info["selected_features"]})
        ens_row = {"split": split_id, "repeat": repeat, "fold": fold, "cv_mode": cv_mode, "model": "weighted_ensemble",
                   "n_features_final": len(ens_selected), "inner_cv_auc": np.mean([i["inner_cv_auc"] for _, i in members]),
                   "ensemble_members": ",".join(n for n, _ in members), **ens_metrics}
        split_rows.append(ens_row)
        full_records.append({"row": ens_row, "selected_features": ens_selected,
                              "best_params": {"ensemble_members": [n for n, _ in members]}})
        for i, ridx in enumerate(val_idx):
            oof_records.append({"row_id": row_ids[ridx], "repeat": repeat, "model": "weighted_ensemble",
                                 "proba": float(ens_proba[i]), "y_true": int(y_val[i])})
        rows.extend(split_rows)

        auc_text = " | ".join(f"{r['model']}={r['roc_auc']:.3f}" for r in split_rows
                               if r["model"] in ("rf", "extra_trees", "xgb", "weighted_ensemble"))
        print(f"[repeat {repeat} fold {fold}] features={ens_row['n_features_final']} | {auc_text}")
    return pd.DataFrame(rows), full_records, pd.DataFrame(oof_records)


## Cell 10b — CV Reporting: Summaries, Subgroup Breakdown, Leakage Ablation

In [ ]:
def confidence_interval(values, confidence: float = 0.95) -> tuple:
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    mean = float(np.mean(values))
    if len(values) == 1:
        return mean, np.nan, np.nan
    sem = np.std(values, ddof=1) / math.sqrt(len(values))
    margin = t.ppf((1 + confidence) / 2, df=len(values) - 1) * sem
    return mean, float(mean - margin), float(mean + margin)


def summarize_metrics(cv_df: pd.DataFrame, outdir: Path) -> pd.DataFrame:
    metric_cols = ["roc_auc", "pr_auc", "f1", "mcc", "sensitivity", "specificity", "balanced_accuracy"]
    rows = []
    for model_name, group in cv_df.groupby("model"):
        row = {"model": model_name, "n_splits": int(len(group))}
        for metric in metric_cols:
            mean, low, high = confidence_interval(group[metric].to_numpy())
            row[f"{metric}_mean"], row[f"{metric}_ci95_low"], row[f"{metric}_ci95_high"] = mean, low, high
            row[f"{metric}_std"] = float(group[metric].std())
        rows.append(row)
    summary = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
    summary.to_csv(outdir / "metric_summary_ci95.csv", index=False)
    return summary


def feature_selection_stability(full_records: list, outdir: Path) -> pd.DataFrame:
    main_records = [r for r in full_records if r["row"]["model"] != "weighted_ensemble"]
    all_feats = sorted({f for r in main_records for f in r["selected_features"]})
    counts = pd.Series(0, index=all_feats, dtype=int)
    for record in main_records:
        counts.loc[record["selected_features"]] += 1
    denom = max(len(main_records), 1)
    stability = (counts / denom).sort_values(ascending=False).rename("selection_frequency").to_frame()
    stability["family"] = [feature_family(f) for f in stability.index]
    stability.index.name = "feature"
    stability.to_csv(outdir / "feature_selection_stability.csv")
    return stability


def subgroup_performance_report(df: pd.DataFrame, oof_df: pd.DataFrame,
                                 subgroup_cols=("MHC_Class_Used", "Organism"),
                                 model_name: str = "weighted_ensemble",
                                 min_group_size: int = 10) -> pd.DataFrame:

    sub = oof_df[oof_df["model"] == model_name].copy()
    agg = sub.groupby("row_id").agg(proba=("proba", "mean"), y_true=("y_true", "first"),
                                     n_folds=("proba", "size")).reset_index()
    meta = df.reset_index(drop=True).reset_index().rename(columns={"index": "row_id"})
    keep_cols = ["row_id"] + [c for c in subgroup_cols if c in meta.columns]
    merged = agg.merge(meta[keep_cols], on="row_id", how="left")

    rows = []
    for col in subgroup_cols:
        if col not in merged.columns:
            continue
        for val, g in merged.groupby(col):
            if g["y_true"].nunique() < 2 or len(g) < min_group_size:
                continue
            rows.append({"subgroup_column": col, "subgroup_value": val, "n": int(len(g)),
                         "n_positive": int((g["y_true"] == 1).sum()),
                         "n_negative": int((g["y_true"] == 0).sum()),
                         "roc_auc": float(roc_auc_score(g["y_true"], g["proba"]))})
    out = pd.DataFrame(rows).sort_values(["subgroup_column", "roc_auc"], ascending=[True, False])
    return out


def quick_leakage_ablation(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config) -> dict:

    probe_kwargs = {**asdict(cfg), "random_search_iter": 10, "outer_repeats": 1}

    print("[leakage ablation] Running HONEST (similarity-group-aware) CV probe...")
    honest_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "always"})
    honest_df, _, _ = run_nested_cv(X, y, groups, honest_cfg)
    honest_auc = float(honest_df.loc[honest_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    print("[leakage ablation] Running NAIVE (non-grouped) CV probe -- for reporting the gap ONLY...")
    naive_cfg = Config(**{**probe_kwargs, "use_similarity_cv": "never"})
    naive_df, _, _ = run_nested_cv(X, y, groups, naive_cfg)
    naive_auc = float(naive_df.loc[naive_df["model"] == "weighted_ensemble", "roc_auc"].mean())

    result = {
        "honest_group_aware_auc_probe": honest_auc,
        "naive_non_grouped_auc_probe": naive_auc,
        "estimated_leakage_inflation": naive_auc - honest_auc,
        "note": "These are LIGHT-BUDGET probes (random_search_iter=10, outer_repeats=1) for "
                "quantifying leakage only. Report cv_fold_results.csv / metric_summary_ci95.csv "
                "(full-budget, similarity-group-aware) as the model's actual performance.",
    }
    print(f"[leakage ablation] {result}")
    return result


## Cell 11 — Final Model, Main Pipeline & Inference

In [ ]:
def train_final_model(X: pd.DataFrame, y: np.ndarray, groups: np.ndarray, cfg: Config):

    class_counts = np.bincount(y.astype(int), minlength=2)
    n_splits = min(cfg.inner_folds, int(class_counts.min()))
    if n_splits < 2:
        raise ValueError(f"Final tuning needs >=2 samples per class; found {int(class_counts.min())} in minority class.")

    if len(np.unique(groups)) >= n_splits:
        cv, cv_groups = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), groups
    else:
        cv, cv_groups = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=cfg.random_seed), None

    tuned = tune_models(X, y, cv, cfg, groups_train=cv_groups)
    members = ensemble_member_items(tuned, cfg.ensemble_min_inner_auc)
    weights = ensemble_weights(members)
    estimators = {name: info["estimator"] for name, info in members}
    threshold = float(np.mean([info["threshold"] for _, info in members]))
    final_model = WeightedSoftVotingEnsemble(estimators=estimators, weights=weights, threshold=threshold)
    return final_model, tuned


def predict_new_peptides(new_csv: str, model_pkl: str, output_csv: str = None) -> pd.DataFrame:
    """Load a final_model.pkl artifact and score new peptides (future validation)."""
    with open(model_pkl, "rb") as f:
        artifact = pickle.load(f)
    model = artifact["model"]
    feature_columns = artifact["feature_columns"]
    cfg = Config(**artifact["config"])

    df = pd.read_csv(new_csv)
    if cfg.label_col not in df.columns:
        df[cfg.label_col] = 0  # placeholder only; not used for scoring

    feat_df = build_feature_matrix(df, cfg)
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    for col in feature_columns:
        if col not in X.columns:
            X[col] = 0.0
    X = X[feature_columns]

    proba = model.predict_proba(X)[:, 1]
    pred = (proba >= model.threshold).astype(int)
    out = feat_df[[c for c in [cfg.id_col, cfg.seq_col] if c in feat_df.columns]].copy()
    out["predicted_probability"] = proba
    out["predicted_label"] = pred
    if output_csv:
        out.to_csv(output_csv, index=False)
        print(f"[predict] saved {len(out)} predictions to {output_csv}")
    return out


## Cell 12 — SHAP Explainability

In [ ]:
def compute_shap_explanations(X: pd.DataFrame, y: np.ndarray, cfg: Config, outdir: Path):

    if not HAS_SHAP:
        print("[shap] shap not installed (`pip install shap`); skipping.")
        return None

    if HAS_XGB:
        base_estimator = XGBClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs, eval_metric="auc",
                                        tree_method="hist", n_estimators=400, max_depth=4,
                                        learning_rate=0.05, subsample=0.8, colsample_bytree=0.7)
        model_tag = "xgb"
    else:
        base_estimator = RandomForestClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs,
                                                  class_weight="balanced_subsample",
                                                  n_estimators=600, max_depth=10)
        model_tag = "rf"

    pipe = make_model_pipeline(base_estimator, cfg)
    pipe.fit(X, y)
    selected = pipeline_selected_features(pipe)

    X_sel = pipe.named_steps["var_corr"].transform(X)
    X_sel = pipe.named_steps["mi"].transform(X_sel)
    X_scaled = pipe.named_steps["scaler"].transform(X_sel)
    X_scaled_df = pd.DataFrame(X_scaled, columns=selected, index=X.index)

    n_bg = min(cfg.shap_max_background, len(X_scaled_df))
    rng = np.random.RandomState(cfg.random_seed)
    sample_idx = rng.choice(len(X_scaled_df), n_bg, replace=False)
    X_sample = X_scaled_df.iloc[sample_idx]

    explainer = shap.TreeExplainer(pipe.named_steps["classifier"])
    shap_values = explainer.shap_values(X_sample)
    if isinstance(shap_values, list):
        shap_values = shap_values[1]
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    shap_df = pd.DataFrame(shap_values, columns=selected)
    shap_df.to_csv(outdir / "shap_values.csv", index=False)
    mean_abs = shap_df.abs().mean().sort_values(ascending=False)
    mean_abs.to_csv(outdir / "shap_feature_importance.csv", header=["mean_abs_shap"])

    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        shap.summary_plot(shap_values, X_sample, show=False, max_display=20)
        plt.tight_layout()
        plt.savefig(outdir / "shap_summary_plot.png", dpi=200, bbox_inches="tight")
        plt.close()
        print(f"[shap] saved shap_summary_plot.png")
    except Exception as e:
        print(f"[shap] plot generation failed (non-fatal): {e}")

    with open(outdir / "shap_explainer_model.pkl", "wb") as f:
        pickle.dump({"pipe": pipe, "explainer_model_tag": model_tag,
                     "selected_features": selected, "config": asdict(cfg)}, f)

    print(f"[shap] explainer model: {model_tag} | top 10 features by mean |SHAP|:")
    print(mean_abs.head(10).to_string())
    return {"explainer_model": model_tag, "top_features": mean_abs.head(20).to_dict()}


## Cell 13 — LIME Explainability

In [ ]:
def compute_lime_explanations(X: pd.DataFrame, y: np.ndarray, cfg: Config, outdir: Path,
                               shap_model_path: Path = None):

    if not HAS_LIME:
        print("[lime] lime not installed (`pip install lime`); skipping.")
        return None

    if shap_model_path is not None and Path(shap_model_path).exists():
        with open(shap_model_path, "rb") as f:
            bundle = pickle.load(f)
        pipe, selected = bundle["pipe"], bundle["selected_features"]
    else:
        base_estimator = (XGBClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs, eval_metric="auc",
                                         tree_method="hist", n_estimators=400, max_depth=4,
                                         learning_rate=0.05, subsample=0.8, colsample_bytree=0.7)
                           if HAS_XGB else
                           RandomForestClassifier(random_state=cfg.random_seed, n_jobs=cfg.n_jobs,
                                                   class_weight="balanced_subsample",
                                                   n_estimators=600, max_depth=10))
        pipe = make_model_pipeline(base_estimator, cfg)
        pipe.fit(X, y)
        selected = pipeline_selected_features(pipe)

    X_sel_all = pipe.named_steps["var_corr"].transform(X)
    X_sel_all = pipe.named_steps["mi"].transform(X_sel_all)
    X_scaled_all = pipe.named_steps["scaler"].transform(X_sel_all)
    X_scaled_all = np.asarray(X_scaled_all)

    classifier = pipe.named_steps["classifier"]
    predict_fn = lambda arr: classifier.predict_proba(arr)

    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_scaled_all, feature_names=selected,
        class_names=["non_immunogenic", "immunogenic"], mode="classification",
        random_state=cfg.random_seed)

    proba = pipe.predict_proba(X)[:, 1]
    rng = np.random.RandomState(cfg.random_seed)
    tp_idx = np.where((y == 1) & (proba > 0.65))[0]
    tn_idx = np.where((y == 0) & (proba < 0.35))[0]
    borderline_idx = np.where(np.abs(proba - 0.5) < 0.08)[0]

    picks = []
    for pool, tag in [(tp_idx, "confident_positive"), (tn_idx, "confident_negative"),
                       (borderline_idx, "borderline")]:
        if len(pool) > 0:
            picks.append((int(rng.choice(pool)), tag))

    results = []
    for idx, tag in picks[:cfg.lime_n_explanations]:
        exp = explainer.explain_instance(X_scaled_all[idx], predict_fn, num_features=10)
        html_path = outdir / f"lime_{tag}_row{idx}.html"
        try:
            exp.save_to_file(str(html_path))
        except Exception as e:
            print(f"[lime] could not save html for row {idx} (non-fatal): {e}")
        results.append({"row_index": idx, "tag": tag, "true_label": int(y[idx]),
                         "predicted_proba": float(proba[idx]), "explanation": exp.as_list()})

    with open(outdir / "lime_explanations.json", "w") as f:
        json.dump(results, f, indent=2)
    print(f"[lime] saved {len(results)} local explanations to lime_explanations.json")
    return results


## Cell 14 — Main Pipeline Runner

In [ ]:
def run_full_pipeline(cfg: Config):
    outdir = Path(cfg.outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    (outdir / "config.json").write_text(json.dumps(asdict(cfg), indent=2))

    # 1. Load data
    df = pd.read_csv(cfg.input_csv)
    print(f"[load] {len(df)} rows from {cfg.input_csv}")

    # 2. Restrict to MHC assay-supported peptides
    df, restrict_report = restrict_to_assay_supported(df, cfg)

    # 2b. NEW: restrict to direct T-cell-assay-confirmed peptides
    df, tcell_report = restrict_to_tcell_assay(df, cfg)

    # 3. Similarity clustering (on the FINAL filtered set)
    seqs = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    groups = compute_similarity_groups(seqs, cfg.similarity_threshold)

    # 4. Resolve label conflicts
    if cfg.resolve_conflicts:
        df, groups, conflict_report = resolve_label_conflicts(df, groups, cfg)
    else:
        conflict_report = {"resolve_conflicts": False}

    # 5. Drop invalid sequences, keep groups aligned
    seqs_clean = df[cfg.seq_col].astype(str).map(clean_sequence)
    valid_mask = seqs_clean.map(is_valid_peptide)
    if (~valid_mask).any():
        n_inv = int((~valid_mask).sum())
        print(f"[pipeline] Dropping {n_inv} invalid sequences before feature extraction.")
        df = df.loc[valid_mask].reset_index(drop=True)
        groups = groups[valid_mask.to_numpy()]

    row_ids = np.arange(len(df))  # stable ids for OOF/subgroup reporting

    # 6. Build feature matrix (ESM-2 cached automatically)
    feat_df = build_feature_matrix(df, cfg)
    if feat_df.empty:
        raise ValueError("No valid peptides remaining after filtering.")

    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    y = feat_df[cfg.label_col].to_numpy().astype(int)
    groups = groups[:len(feat_df)]
    row_ids = row_ids[:len(feat_df)]

    # 7. Nested, similarity-aware CV  (the HONEST performance estimate)
    cv_df, full_records, oof_df = run_nested_cv(X, y, groups, cfg, row_ids=row_ids)
    cv_df.to_csv(outdir / "cv_fold_results.csv", index=False)
    oof_df.to_csv(outdir / "oof_predictions.csv", index=False)
    summary = summarize_metrics(cv_df, outdir)
    feature_selection_stability(full_records, outdir)

    # 7b. NEW: honest subgroup breakdown (MHC class, organism if present)
    subgroup_cols = [c for c in ["MHC_Class_Used", "Organism"] if c in df.columns]
    subgroup_df = pd.DataFrame()
    if subgroup_cols:
        subgroup_df = subgroup_performance_report(df, oof_df, subgroup_cols=subgroup_cols)
        subgroup_df.to_csv(outdir / "subgroup_performance.csv", index=False)
        print("\n[subgroup performance]\n" + subgroup_df.to_string(index=False))

    # 8. Final deployable model (fit on all resolved data; NOT the perf estimate)
    final_model, final_tuned = train_final_model(X, y, groups, cfg)
    with open(outdir / "final_model.pkl", "wb") as f:
        pickle.dump({
            "model": final_model,
            "feature_columns": X.columns.tolist(),
            "config": asdict(cfg),
            "member_inner_cv_auc": {n: i["inner_cv_auc"] for n, i in final_tuned.items()},
        }, f)
    print(f"[save] final_model.pkl written to {outdir}")

    # 9. NEW: explainability artifacts for the paper (SHAP + LIME)
    shap_result, lime_result = None, None
    if cfg.save_explainability_artifacts:
        shap_result = compute_shap_explanations(X, y, cfg, outdir)
        lime_result = compute_lime_explanations(
            X, y, cfg, outdir, shap_model_path=outdir / "shap_explainer_model.pkl")

    # 10. Report
    ens_auc = None
    if "weighted_ensemble" in summary["model"].values:
        ens_auc = float(summary.loc[summary["model"] == "weighted_ensemble", "roc_auc_mean"].iloc[0])
    report = {
        "restrict_to_assay_support": restrict_report,
        "restrict_to_tcell_assay": tcell_report,
        "conflict_resolution": conflict_report,
        "n_features": int(X.shape[1]),
        "n_peptides_final": int(len(feat_df)),
        "ensemble_roc_auc_mean": ens_auc,
        "shap_computed": shap_result is not None,
        "lime_computed": lime_result is not None,
    }
    (outdir / "run_report.json").write_text(json.dumps(report, indent=2))

    print(f"\n[done] {report}")
    print("\n" + "=" * 70)
    print(summary.to_string(index=False))
    print("=" * 70)

    return {
        "df": df, "X": X, "y": y, "groups": groups, "cv_df": cv_df,
        "oof_df": oof_df, "summary": summary, "subgroup_df": subgroup_df,
        "final_model": final_model, "shap_result": shap_result, "lime_result": lime_result,
        "report": report,
    }


## Cell 15 — Leakage Ablation Trigger

In [ ]:
def run_leakage_ablation_report(cfg: Config, outdir: Path):

    df = pd.read_csv(cfg.input_csv)
    df, _ = restrict_to_assay_supported(df, cfg)
    df, _ = restrict_to_tcell_assay(df, cfg)
    seqs = df[cfg.seq_col].astype(str).map(clean_sequence).tolist()
    groups = compute_similarity_groups(seqs, cfg.similarity_threshold)
    if cfg.resolve_conflicts:
        df, groups, _ = resolve_label_conflicts(df, groups, cfg)
    valid_mask = df[cfg.seq_col].astype(str).map(clean_sequence).map(is_valid_peptide)
    df = df.loc[valid_mask].reset_index(drop=True)
    groups = groups[valid_mask.to_numpy()][:len(df)]

    feat_df = build_feature_matrix(df, cfg)
    drop_cols = [c for c in [cfg.id_col, cfg.label_col, cfg.seq_col] if c in feat_df.columns]
    X = feat_df.drop(columns=drop_cols)
    y = feat_df[cfg.label_col].to_numpy().astype(int)
    groups = groups[:len(feat_df)]

    result = quick_leakage_ablation(X, y, groups, cfg)
    (Path(outdir) / "leakage_ablation.json").write_text(json.dumps(result, indent=2))
    return result


## Cell 16 — Colab / environment convenience

In [ ]:
def maybe_colab_upload(default_filename: str):

    if Path(default_filename).exists():
        return
    try:
        from google.colab import files  # type: ignore
        print(f"[colab] '{default_filename}' not found -- please upload it:")
        files.upload()
    except ImportError:
        pass  # not running in Colab; assume the file is already on disk


def zip_and_offer_download(outdir: str):

    import shutil
    zip_path = shutil.make_archive(outdir, "zip", root_dir=outdir)
    print(f"[package] wrote {zip_path}")
    try:
        from google.colab import files  # type: ignore
        files.download(zip_path)
    except ImportError:
        print(f"[package] (not in Colab) find your results at: {zip_path}")


## Cell 17 — Configuration & Run (edit the path above, then run)



In [ ]:
cfg = Config(
    input_csv=r"H:\Peptide\main_dataset_cleaned_2.csv",
    outdir="ml_report_cleaned2_q1_final_esm150m",

    n_jobs=4,

    use_esm_embeddings=True,
    esm_model_name="esm2_t30_150M_UR50D",
    random_search_iter=35,
    outer_repeats=3,
    outer_folds=5,

    require_tcell_assay=True,
    use_similarity_cv="auto",
    save_explainability_artifacts=True,
)

results = run_full_pipeline(cfg)

## Cell 18 — Leakage ablation for your manuscript



In [ ]:
ablation = run_leakage_ablation_report(cfg, Path(cfg.outdir))
print(ablation)


## Cell 19 — Score brand-new peptides later



In [ ]:
from pathlib import Path
import pickle
from dataclasses import asdict

model_path = Path(cfg.outdir) / "final_model.pkl"

if model_path.exists():
    print(f"Already saved: {model_path.resolve()}")
    print(f"  size: {model_path.stat().st_size / 1024:.1f} KB")
else:
    print(f"{model_path} not found on disk -- saving now from 'results' in memory.")
    if "results" not in globals():
        raise NameError(
            "No 'results' variable in memory and no file on disk. "
            "Rerun Cell 42 (`results = run_full_pipeline(cfg)`) first -- there's no "
            "shortcut to the saved model without that."
        )
    Path(cfg.outdir).mkdir(parents=True, exist_ok=True)
    with open(model_path, "wb") as f:
        pickle.dump({
            "model": results["final_model"],
            "feature_columns": results["X"].columns.tolist(),
            "config": asdict(cfg),
        }, f)
    print(f"Saved to {model_path.resolve()}")

Already saved: C:\Users\Partho Bosu\ml_report_cleaned2_q1_final_esm150m\final_model.pkl
  size: 5615.6 KB


#Cell-19 Saves ROC curve, PR curve, confusion matrix, SHAP plots, LIME plots, and the subgroup performance chart

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, precision_recall_curve as pr_curve_fn, auc, ConfusionMatrixDisplay

def generate_publication_figures(results: dict, cfg: Config):

    figdir = Path(cfg.outdir) / "figures"
    figdir.mkdir(parents=True, exist_ok=True)

    oof_df = results["oof_df"]
    ens = (oof_df[oof_df["model"] == "weighted_ensemble"]
           .groupby("row_id").agg(proba=("proba", "mean"), y_true=("y_true", "first")).reset_index())
    y_true, proba = ens["y_true"].to_numpy(), ens["proba"].to_numpy()

    # 1. ROC curve
    fpr, tpr, _ = roc_curve(y_true, proba)
    roc_auc_val = auc(fpr, tpr)
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"Weighted ensemble (AUC = {roc_auc_val:.3f})", linewidth=2)
    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Random (AUC = 0.500)")
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title("ROC Curve (honest, out-of-fold, similarity-group-aware CV)")
    plt.legend(loc="lower right"); plt.tight_layout()
    plt.savefig(figdir / "roc_curve.png", dpi=300); plt.close()
    print(f"[figures] roc_curve.png saved (AUC={roc_auc_val:.3f})")

    # 2. Precision-Recall curve
    precision, recall, _ = pr_curve_fn(y_true, proba)
    pr_auc_val = average_precision_score(y_true, proba)
    plt.figure(figsize=(6, 6))
    plt.plot(recall, precision, label=f"Weighted ensemble (AP = {pr_auc_val:.3f})", linewidth=2)
    baseline = y_true.mean()
    plt.axhline(baseline, linestyle="--", color="gray", label=f"Baseline (prevalence = {baseline:.3f})")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("Precision-Recall Curve (honest, out-of-fold)")
    plt.legend(loc="lower left"); plt.tight_layout()
    plt.savefig(figdir / "precision_recall_curve.png", dpi=300); plt.close()
    print(f"[figures] precision_recall_curve.png saved (AP={pr_auc_val:.3f})")

    # 3. Confusion matrix (at the OOF-optimal threshold)
    threshold = optimize_threshold(y_true, proba, cfg.threshold_metric)
    pred = (proba >= threshold).astype(int)
    fig, ax = plt.subplots(figsize=(5, 5))
    ConfusionMatrixDisplay.from_predictions(y_true, pred, display_labels=["Non-immunogenic", "Immunogenic"],
                                             cmap="Blues", ax=ax, colorbar=False)
    ax.set_title(f"Confusion Matrix (threshold={threshold:.2f})")
    plt.tight_layout()
    plt.savefig(figdir / "confusion_matrix.png", dpi=300); plt.close()
    print("[figures] confusion_matrix.png saved")

    # 4. SHAP: bar chart of global importance + copy the beeswarm plot already made in Cell 15
    shap_values_path = Path(cfg.outdir) / "shap_values.csv"
    if shap_values_path.exists():
        shap_df = pd.read_csv(shap_values_path)
        mean_abs = shap_df.abs().mean().sort_values(ascending=False).head(20)
        plt.figure(figsize=(7, 6))
        mean_abs.iloc[::-1].plot(kind="barh", color="#4C72B0")
        plt.xlabel("Mean |SHAP value|"); plt.title("SHAP Global Feature Importance (top 20)")
        plt.tight_layout()
        plt.savefig(figdir / "shap_bar_importance.png", dpi=300); plt.close()
        print("[figures] shap_bar_importance.png saved")
    else:
        print("[figures] shap_values.csv not found -- run the SHAP/LIME cell (15) first")

    existing_shap_plot = Path(cfg.outdir) / "shap_summary_plot.png"
    if existing_shap_plot.exists():
        import shutil
        shutil.copy(existing_shap_plot, figdir / "shap_summary_plot.png")
        print("[figures] shap_summary_plot.png copied into figures/")

    # 5. LIME plots as PNGs, rebuilt from the saved JSON (no need to re-run LIME)
    lime_json_path = Path(cfg.outdir) / "lime_explanations.json"
    if lime_json_path.exists():
        with open(lime_json_path) as f:
            lime_results = json.load(f)
        for entry in lime_results:
            expl = entry["explanation"]
            feats = [e[0] for e in expl][::-1]
            weights = [e[1] for e in expl][::-1]
            colors = ["#d62728" if w < 0 else "#2ca02c" for w in weights]
            plt.figure(figsize=(7, 5))
            plt.barh(feats, weights, color=colors)
            plt.axvline(0, color="black", linewidth=0.8)
            plt.title(f"LIME local explanation -- {entry['tag']} "
                      f"(true={entry['true_label']}, predicted_p={entry['predicted_proba']:.2f})")
            plt.xlabel("Contribution to predicted probability")
            plt.tight_layout()
            fname = f"lime_{entry['tag']}_row{entry['row_index']}.png"
            plt.savefig(figdir / fname, dpi=300); plt.close()
            print(f"[figures] {fname} saved")
    else:
        print("[figures] lime_explanations.json not found -- run the SHAP/LIME cell (15) first")

    # 6. Subgroup performance (MHC class / organism) as a bar chart
    subgroup_df = results.get("subgroup_df", pd.DataFrame())
    if subgroup_df is not None and not subgroup_df.empty:
        n_cols = subgroup_df["subgroup_column"].nunique()
        fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
        if n_cols == 1:
            axes = [axes]
        for ax, (col, g) in zip(axes, subgroup_df.groupby("subgroup_column")):
            g = g.sort_values("roc_auc")
            ax.barh(g["subgroup_value"].astype(str), g["roc_auc"], color="#4C72B0")
            ax.axvline(0.5, color="gray", linestyle="--")
            ax.set_xlabel("ROC-AUC"); ax.set_title(col); ax.set_xlim(0, 1)
        plt.tight_layout()
        plt.savefig(figdir / "subgroup_performance.png", dpi=300); plt.close()
        print("[figures] subgroup_performance.png saved")

    print(f"\nAll figures saved to: {figdir.resolve()}")
    return figdir


generate_publication_figures(results, cfg)

# Cell-20 Download all things

In [ ]:
import shutil

model_src = Path(cfg.outdir) / "final_model.pkl"
model_dest = Path(cfg.outdir) / "figures" / "final_model.pkl"

if model_src.exists():
    shutil.copy2(model_src, model_dest)
    print(f"[model] copied final_model.pkl into figures/ ({model_dest.stat().st_size / 1024:.0f} KB)")
else:
    print(f"[model] {model_src} not found -- make sure Cell 17 (run_full_pipeline) finished successfully first")

zip_path = shutil.make_archive(str(Path(cfg.outdir) / "model_and_figures"), "zip",
                                root_dir=Path(cfg.outdir) / "figures")
print(f"[model] zipped everything to: {zip_path}")

# CELL 21 — Confirm ESM-2 embeddings are actually present in the reported result

In [ ]:

esm_cols = [c for c in results["X"].columns if c.startswith("ESM2_")]
non_esm_cols = [c for c in results["X"].columns if not c.startswith("ESM2_")]

print(f"Total features in reported model:      {results['X'].shape[1]}")
print(f"ESM-2 embedding columns present:       {len(esm_cols)}")
print(f"Non-ESM (classical + MHC) columns:     {len(non_esm_cols)}")
print(f"cfg.use_esm_embeddings was set to:     {cfg.use_esm_embeddings}")
print(f"cfg.esm_model_name:                    {cfg.esm_model_name}")

if cfg.use_esm_embeddings and len(esm_cols) == 0:
    print("\n*** WARNING: use_esm_embeddings=True but NO ESM2_ columns found. ***")
    print("*** ESM-2 silently failed -- your reported AUC does NOT include it. ***")
elif not cfg.use_esm_embeddings:
    print("\ncfg.use_esm_embeddings was False -- ESM-2 intentionally not used.")
else:
    print(f"\nConfirmed: ESM-2 ({cfg.esm_model_name}) IS included "
          f"({len(esm_cols)}/{results['X'].shape[1]} = "
          f"{100*len(esm_cols)/results['X'].shape[1]:.1f}% of the feature space).")


outdir = Path(cfg.outdir)
try:
    stability = pd.read_csv(outdir / "feature_selection_stability.csv")
    esm_stability = stability[stability["family"] == "ESM2"]
    if len(esm_stability):
        print(f"\nESM2 features selected in >=1 CV fold: {len(esm_stability)}")
        print(f"Mean selection frequency: {esm_stability['selection_frequency'].mean():.3f}")
        print("Top 5 most stable ESM2 features:")
        print(esm_stability.head(5).to_string(index=False))
    else:
        print("\nESM2 columns exist in X but were NEVER selected by the MI/variance filter "
              "in any CV fold -- they were filtered out before reaching the classifier.")
except FileNotFoundError:
    print(f"\n(feature_selection_stability.csv not found in {outdir}.)")

diagnostic = {
    "n_total_features": int(results["X"].shape[1]),
    "n_esm_features": len(esm_cols),
    "esm_confirmed_in_use": bool(cfg.use_esm_embeddings and len(esm_cols) > 0),
}
(outdir / "esm_confirmation_check.json").write_text(json.dumps(diagnostic, indent=2))
print(f"\nSaved to: {outdir / 'esm_confirmation_check.json'}")

# CELL 22 — Is any model actually better, or is it noise across folds?

In [ ]:

from scipy.stats import wilcoxon
from itertools import combinations

def bh_fdr(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    critical = (np.arange(1, n + 1) / n) * alpha
    passed = pvals[order] <= critical
    sig_sorted = np.zeros(n, dtype=bool)
    if passed.any():
        sig_sorted[: np.max(np.where(passed)[0]) + 1] = True
    significant = np.zeros(n, dtype=bool)
    significant[order] = sig_sorted
    return significant

cv_df = results["cv_df"].copy()
pivot = cv_df.pivot_table(index="split", columns="model", values="roc_auc")
models = pivot.columns.tolist()
print(f"Paired comparison across {pivot.shape[0]} CV splits, {len(models)} models.\n")

rows = []
for m1, m2 in combinations(models, 2):
    paired = pivot[[m1, m2]].dropna()
    diff = paired[m1] - paired[m2]
    if len(paired) < 3 or (diff == 0).all():
        stat, p = np.nan, 1.0
    else:
        stat, p = wilcoxon(paired[m1], paired[m2], zero_method="wilcox", alternative="two-sided")
    rows.append({
        "model_a": m1, "model_b": m2,
        "mean_auc_a": paired[m1].mean(), "mean_auc_b": paired[m2].mean(),
        "mean_diff_a_minus_b": diff.mean(), "n_paired_folds": len(paired),
        "wilcoxon_stat": stat, "p_value": p,
    })

pairwise_df = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
pairwise_df["significant_fdr05"] = bh_fdr(pairwise_df["p_value"].to_numpy())

outdir = Path(cfg.outdir)
pairwise_df.to_csv(outdir / "model_pairwise_significance.csv", index=False)

print(pairwise_df[["model_a", "model_b", "mean_auc_a", "mean_auc_b", "mean_diff_a_minus_b",
                    "n_paired_folds", "p_value", "significant_fdr05"]].to_string(index=False))
n_sig = int(pairwise_df["significant_fdr05"].sum())
print(f"\n{n_sig} of {len(pairwise_df)} pairwise comparisons significant after BH-FDR (alpha=0.05).")
print(f"Saved to: {outdir / 'model_pairwise_significance.csv'}")

# CELL 23 — Class-imbalance robustness check

In [ ]:
rng = np.random.RandomState(cfg.random_seed)

oof = results["oof_df"]
ens = (oof[oof["model"] == "weighted_ensemble"]
       .groupby("row_id").agg(proba=("proba", "mean"), y_true=("y_true", "first")).reset_index())
y_true_all = ens["y_true"].to_numpy()
proba_all = ens["proba"].to_numpy()
pos_idx = np.where(y_true_all == 1)[0]
neg_idx = np.where(y_true_all == 0)[0]
print(f"OOF pool: n_pos={len(pos_idx)}, n_neg={len(neg_idx)}, "
      f"natural prevalence={len(pos_idx)/(len(pos_idx)+len(neg_idx)):.3f}\n")

fixed_threshold = optimize_threshold(y_true_all, proba_all, cfg.threshold_metric)
print(f"Decision threshold fixed at: {fixed_threshold:.3f}\n")

ratios = [1, 2, 5, 10, 20]
n_bootstrap = 500
sample_size = 200


rows = []
for neg_per_pos in ratios:
    n_pos_t = max(2, round(sample_size / (1 + neg_per_pos)))
    n_neg_t = sample_size - n_pos_t
    boot = []
    for _ in range(n_bootstrap):
        idx = np.concatenate([rng.choice(pos_idx, n_pos_t, replace=True),
                               rng.choice(neg_idx, n_neg_t, replace=True)])
        boot.append(metric_row(y_true_all[idx], proba_all[idx], fixed_threshold))
    bm = pd.DataFrame(boot)
    row = {"ratio": f"1:{neg_per_pos}", "prevalence": n_pos_t / sample_size}
    for metric in ["roc_auc", "pr_auc", "f1", "mcc", "sensitivity", "specificity", "balanced_accuracy"]:
        mean, low, high = confidence_interval(bm[metric].to_numpy())
        row[f"{metric}_mean"], row[f"{metric}_ci95_low"], row[f"{metric}_ci95_high"] = mean, low, high
    rows.append(row)

imbalance_df = pd.DataFrame(rows)
outdir = Path(cfg.outdir)
imbalance_df.to_csv(outdir / "class_imbalance_robustness.csv", index=False)

print(imbalance_df[["ratio", "prevalence", "roc_auc_mean", "pr_auc_mean", "f1_mean", "mcc_mean"]].to_string(index=False))
print(f"\nSaved to: {outdir / 'class_imbalance_robustness.csv'}")
print("\nROC-AUC should stay roughly flat (prevalence-invariant by construction).")
print("PR-AUC/F1/MCC dropping as the ratio worsens is expected -- that's the result to report.")

# CELL 24 — k-mer / motif enrichment, independent of SHAP

In [ ]:
from scipy.stats import ttest_ind

def bh_fdr(pvals, alpha=0.05):
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    critical = (np.arange(1, n + 1) / n) * alpha
    passed = pvals[order] <= critical
    sig_sorted = np.zeros(n, dtype=bool)
    if passed.any():
        sig_sorted[: np.max(np.where(passed)[0]) + 1] = True
    significant = np.zeros(n, dtype=bool)
    significant[order] = sig_sorted
    return significant

X_full, y_full = results["X"], results["y"]
dpc_cols = [c for c in X_full.columns if c.startswith("DPC_")]
pos_mask, neg_mask = y_full == 1, y_full == 0
print(f"Testing {len(dpc_cols)} dipeptides: immunogenic n={int(pos_mask.sum())}, "
      f"non-immunogenic n={int(neg_mask.sum())}\n")

rows = []
for col in dpc_cols:
    pos_vals = X_full.loc[pos_mask, col].to_numpy()
    neg_vals = X_full.loc[neg_mask, col].to_numpy()
    if pos_vals.std() == 0 and neg_vals.std() == 0:
        continue
    stat, p = ttest_ind(pos_vals, neg_vals, equal_var=False)  # Welch's t-test
    rows.append({"dipeptide": col.replace("DPC_", ""),
                 "mean_freq_immunogenic": pos_vals.mean(),
                 "mean_freq_non_immunogenic": neg_vals.mean(),
                 "mean_diff": pos_vals.mean() - neg_vals.mean(),
                 "t_stat": stat, "p_value": p})

motif_df = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
motif_df["significant_fdr05"] = bh_fdr(motif_df["p_value"].to_numpy())

outdir = Path(cfg.outdir)
motif_df.to_csv(outdir / "dipeptide_motif_enrichment.csv", index=False)

n_sig = int(motif_df["significant_fdr05"].sum())
print(f"{n_sig} of {len(motif_df)} dipeptides significant after BH-FDR (alpha=0.05).\n")
print("Top 15:")
print(motif_df.head(15).to_string(index=False))

# Cross-check against SHAP -- "two independent methods, same signal" narrative
try:
    shap_imp = pd.read_csv(outdir / "shap_feature_importance.csv", index_col=0)
    shap_dpc = shap_imp[shap_imp.index.str.startswith("DPC_")].copy()
    shap_dpc.index = shap_dpc.index.str.replace("DPC_", "", regex=False)
    top_motifs = set(motif_df.loc[motif_df["significant_fdr05"], "dipeptide"])
    top_shap = set(shap_dpc.sort_values(shap_dpc.columns[0], ascending=False).head(15).index)
    overlap = sorted(top_motifs & top_shap)
    print(f"\nOverlap between BH-significant motifs and top-15 SHAP dipeptides: {overlap}")
    print(f"({len(overlap)} dipeptides flagged independently by BOTH t-test AND SHAP)")
except FileNotFoundError:
    print("\n(shap_feature_importance.csv not found -- run the SHAP cell first for this cross-check.)")

print(f"\nSaved to: {outdir / 'dipeptide_motif_enrichment.csv'}")

# CELL 25 — How much does each feature family contribute alone?
 Runtime warning: this reruns a light-budget nested CV once per family. Expect it to take a meaningful fraction of your original Cell 17 runtime.

In [ ]:

X_full, y_full, groups_full = results["X"], results["y"], results["groups"]
families = sorted(set(feature_family(c) for c in X_full.columns))
print(f"Feature families found: {families}\n")

probe_kwargs = {**asdict(cfg), "random_search_iter": 10, "outer_repeats": 1}
rows = []
for fam in families:
    fam_cols = [c for c in X_full.columns if feature_family(c) == fam]
    X_fam = X_full[fam_cols]
    print(f"--- {fam} ({len(fam_cols)} features) ---")
    fam_cfg = Config(**{**probe_kwargs, "mi_top_k": min(probe_kwargs["mi_top_k"], len(fam_cols))})
    try:
        fam_cv_df, _, _ = run_nested_cv(X_fam, y_full, groups_full, fam_cfg)
        ens = fam_cv_df[fam_cv_df["model"] == "weighted_ensemble"]
        mean, low, high = confidence_interval(ens["roc_auc"].to_numpy())
        rows.append({"family": fam, "n_features": len(fam_cols), "roc_auc_mean": mean,
                      "roc_auc_ci95_low": low, "roc_auc_ci95_high": high, "n_splits": len(ens)})
    except Exception as exc:
        print(f"  failed: {exc}")
        rows.append({"family": fam, "n_features": len(fam_cols), "roc_auc_mean": np.nan,
                      "roc_auc_ci95_low": np.nan, "roc_auc_ci95_high": np.nan, "n_splits": 0})

family_df = pd.DataFrame(rows).sort_values("roc_auc_mean", ascending=False)
outdir = Path(cfg.outdir)
family_df.to_csv(outdir / "feature_family_ablation.csv", index=False)

full_auc = results["report"]["ensemble_roc_auc_mean"]
print("\n" + "=" * 70)
print(f"Full model (all families combined): {full_auc:.4f}")
print("=" * 70)
print(family_df.to_string(index=False))
print(f"\nSaved to: {outdir / 'feature_family_ablation.csv'}")
print("Note: light search budget (10 iters, 1 repeat) for speed -- comparative, not headline, numbers.")